### pytorch入门

In [2]:
import torch

x = torch.tensor(1)
print(x)

tensor(1)


### 张量创建

In [ ]:
# 创建形状为 2x4 的张量，数值范围[0,1)
y = torch.rand(2, 4)        # 均匀分布
y0 = torch.randn_like(y)    # 正态分布
print(y, y0)

# 创建全0张量
y1 = torch.zeros(2, 3, dtype=torch.long)
print(y1)

# python列表创建为张量
y2 = torch.tensor([1,2,3], dtype=float)
print(y2)

import numpy as np
y3 = np.array([4,5,6])
y3 = torch.from_numpy(y3)
print(y3)

tensor([[0.1329, 0.1598, 0.8507, 0.8244],
        [0.6942, 0.1677, 0.3760, 0.9568]]) tensor([[-0.5607,  0.8708, -1.2440, -0.4289],
        [ 1.1191,  0.4959,  0.7369, -0.9298]])
tensor([[0, 0, 0],
        [0, 0, 0]])
tensor([1., 2., 3.], dtype=torch.float64)
tensor([4, 5, 6])


### 张量属性

In [15]:
print(y.shape)      # 形状
print(y2.shape)
print(y2.dtype)     # 数据类型
print(y3.device)    # 所在设备

torch.Size([2, 4])
torch.Size([3])
torch.float64
cpu


### 张量计算

In [ ]:
# 张量加法
x = torch.tensor([[1,2,3],[4,5,6]])
y = torch.ones(2,3)
z = x+y     # 等价于 torch.add(x, y, out=z)
print(z)

# 索引，类似于Numpy
# 整数索引降维，切片索引不降维
print(z[:,1])   # 所有行的第一列
print(z[:,:1])

# 展平成为1维向量 以及形状变换
x = x.view(6)
print(x, x.shape)
x = x.reshape(3, 2)
print(x, x.shape)

tensor([[2., 3., 4.],
        [5., 6., 7.]])
tensor([3., 6.])
tensor([[2.],
        [5.]])
tensor([1, 2, 3, 4, 5, 6]) torch.Size([6])
tensor([[1, 2],
        [3, 4],
        [5, 6]]) torch.Size([3, 2])


### 自动求导(Autograd)
```requires_grad=True``` 表示：以后所有由 x 得到的结果，都会形成一张计算图（Computation Graph）  
```y.backward()``` 表示：从 y 开始，沿着计算图反向传播，计算每个变量的梯度。  

##### 学到这里我突然忘了为什么要进行**梯度下降**，后来借助GPT，我想出了下面的一个类比可以辅助理解梯度下降的原理的原因
梯度下降的直观理解：  
一个人（神经网络）想从山顶走到山下（找到损失函数极小值），为了最快地下山他想每次走的时候都走最陡的那个路线（梯度下降方向），  
当下一步没有陡坡的时候（梯度为0）他就可以认为自己已经到了山脚（损失函数达到极小值，模型收敛）。  
在下山的过程中他每次都用当前的坡度（网络参数梯度）来调整自己的徒步路线（网络参数本身），所以到达山脚的时候  
已经相当于完成了下山的路径（完成模型对于数据的拟合）。  
$$
w = w - \eta \frac{\partial L}{\partial w}
$$
<br>  


In [9]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3*x + 1
y.backward()    # 计算梯度
print(x.grad)   # 输出 dy/dx = 2x + 3 = 7

tensor(7.)


### 模型构建
网络通过继承```nn.Module```这个父类，从而有了很多预先设计好的方法可以供用户便捷使用  
如```net.parameters()```实现参数管理；```net.train()、net.eval()```自动使网络中的Dropout、BatchNorm等  
模块进入不同的工作模式，它会递归通知模块们从而实现```Dropout.train()```或者```BatchNorm.eval()```

In [3]:
import torch
import torch.nn as nn

class SimpleNet(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(784, 256)  # 输入到隐藏层
        self.fc2 = nn.Linear(256, 10)   # 隐藏层到输出层
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleNet()


```model(x)与model.forward(x) 等价```

In [ ]:
input_x = torch.randn(5, 784)
pred = model.forward(input_x)
print(pred, pred.shape)
print(pred.device)

tensor([[ 0.2847,  0.4577, -0.3131,  0.0463, -0.1155,  0.0404, -0.0068,  0.1543,
          0.3424,  0.0327],
        [ 0.3826,  0.1841, -0.1930,  0.0606, -0.0839,  0.1201,  0.1453, -0.0535,
          0.1891,  0.0567],
        [ 0.0624,  0.3192, -0.2087,  0.1324,  0.1083, -0.1801,  0.4194, -0.0725,
          0.1973,  0.1385],
        [-0.3543,  0.4501, -0.3015, -0.0499, -0.0396, -0.0534,  0.0640,  0.0824,
          0.2275,  0.2862],
        [-0.1034,  0.5161,  0.2655, -0.3048, -0.0717, -0.1430,  0.1762,  0.0251,
          0.2417, -0.3590]], grad_fn=<AddmmBackward0>) torch.Size([5, 10])
cpu


torch默认在cpu中计算，可以通过```.to(device)```将模型放在**指定gpu**上计算  
```device = torch.device("cuda:0")``` 指定在第0编号的gpu计算

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, type(device))

model_gpu = SimpleNet().to(device)
input_x = torch.randn(5, 784).to(device)
pred = model_gpu.forward(input_x)
print(pred, pred.shape)
print(pred.device)

cuda <class 'torch.device'>
tensor([[ 0.3247,  0.1557,  0.2061,  0.6248, -0.2955,  0.1143, -0.1060, -0.4335,
          0.0918, -0.1414],
        [ 0.0198,  0.0422,  0.4650, -0.1666, -0.1243,  0.2430, -0.3719, -0.1230,
         -0.2726,  0.2925],
        [-0.2981,  0.2927,  0.3591,  0.0897, -0.4003,  0.2671, -0.0933, -0.3486,
         -0.1183, -0.0341],
        [ 0.0658,  0.0476,  0.0061,  0.1222, -0.4160,  0.0439, -0.0910, -0.2469,
         -0.3242,  0.0414],
        [ 0.1863,  0.3547,  0.2935,  0.2210, -0.2779,  0.1033, -0.1206, -0.3198,
          0.0187,  0.0609]], device='cuda:0', grad_fn=<AddmmBackward0>) torch.Size([5, 10])
cuda:0


### 数据处理
```__len__```：告诉 PyTorch 数据集有多少条样本。  
```__getitem__```：告诉 PyTorch 如何根据索引取出一条样本。  

```DataLoader```根据参数配置（如批次大小）来从输入的dataset中取出指定数量的数据进行训练

In [ ]:

from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

dataset = CustomDataset(data, labels)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)


训练和验证数据集划分后进行加载

In [ ]:
train_dataset = CustomDataset(train_data, train_labels)
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)

val_dataset = CustomDataset(val_data, val_labels)
val_dataloader = DataLoader(val_dataset, batch_size=64, shuffle=False)

### 训练
```loss.backward()```是计算梯度的过程，在此之前要把上一次的梯度清空。参数梯度结果保存到了```.grad()```中  
所以先进行```optimizer.zero_grad()```的操作，将模型的上一次梯度结果给清空  
否则，当前梯度会与上一轮**梯度累加**，从而导致参数更新错误。  

参数的梯度存在：```model.fc1.weight.grad```  

```loss.item()```把只有一个元素（标量）的 Tensor 转换成 Python 的 float。

In [ ]:
# 损失函数
criterion = nn.CrossEntropyLoss()
# 优化器
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

iteration = 10
for epoch in range(iteration):
    model.train()   # 设置为训练模式
    train_loss = 0
    for inputs, labels in train_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        # 前向传播
        pred = model(inputs)
        loss = criterion(pred, labels)
        train_loss += loss.item()

        # 反向传播与优化
        optimizer.zero_grad()   # 清空上一次梯度
        loss.backward()         # 计算梯度
        optimizer.step()        # 更新参数
    
    train_loss /= len(train_dataloader)
    
    # 验证
    model.eval()
    with torch.no_grad():
        val_loss = 0
        # ...计算验证集损失...
        for inputs, labels in val_dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            # 前向传播
            pred = model(inputs)
            loss = criterion(pred, labels)
            val_loss += loss.item()
        
        val_loss /= len(val_dataloader)